In [ ]:
from behavior_utils import generate_behavior_summary_combined
from general_utils import find_ephys_sessions
from general_utils import smart_read_csv
import pandas as pd
from ephys_behavior import (
    get_the_mean_firing_rate_combined,   # ← updated name here
    correlate_firing_latent_multiple_variable
)
import os

# 1) find your sorted sessions
_, _, spike_sorted_sessions = find_ephys_sessions()
print("Sessions to process:", spike_sorted_sessions)


variables = [
["QLearning_L2F1_softmax-reward","QLearning_L2F1_softmax-chosenQ-1"],
[ "QLearning_L2F1_softmax-reward","QLearning_L2F1_softmax-sumQ-1"],
["ForagingCompareThreshold-reward","ForagingCompareThreshold-value-1"],
["QLearning_L2F1_softmax-reward"],
["QLearning_L2F1_softmax-chosenQ-1"],
["QLearning_L2F1_softmax-sumQ-1"],
["ForagingCompareThreshold-reward"],
["ForagingCompareThreshold-value-1"],
["QLearning_L2F1_softmax-RPE"],
["ForagingCompareThreshold-RPE"],
]

# 3) ensure results folder
results_folder = '/root/capsule/scratch/correlation_results/'
os.makedirs(results_folder, exist_ok=True)

# 4) iterate sessions
for sess in spike_sorted_sessions:
    try:
        print(f"\n=== Processing session {sess} ===")

        # a) behavior summary
        generate_behavior_summary_combined(
            session_names=[sess],
            save_result=True,
            save_folder=results_folder,
            save_name=f"behavior_summary-{sess}.csv"
        )

        # b) firing rates (note the new call here)
        get_the_mean_firing_rate_combined(
            session_names=[sess],
            save_result=True,
            z_score = False,
            save_folder=results_folder,
            save_name=f"firing_rates-{sess}.csv"
        )
        df_combined_firing_rates    = smart_read_csv(os.path.join(results_folder, f"firing_rates-{sess}.csv"))
        df_combined_behavior_summary = smart_read_csv(os.path.join(results_folder, f"behavior_summary-{sess}.csv"))
        # c) correlations
        mv_df = correlate_firing_latent_multiple_variable(
            df_combined_firing_rates,
            df_combined_behavior_summary,
            variables=variables,
            correlation_model=[ "ARDL_model",'simple_LR'],
            save_result=True,
            save_name=f"correlations_multi-{sess}"
        )

    except Exception as e:
        print(f"!! Error in session {sess}: {e}")
        continue
